#### Workflow Objectives: 
The primary focus of this phase is to identify regions of open chromatin and charactere their genomic locations. 

* **Peak Calling**: Identify chromatin accessibility peaks using MACS2 (v2.2.6) in BAMPE mode.
* **Peak QC**: Evaluate peak reliability with ChIPQC (v1.40.0), focusing on the "Fraction of Reads in Peaks" (FRiP) and avoiding ENCODE blacklist regions.
* **Genomic Annotation**: Use ChIPseeker (v1.40.0) to map peaks to the nearest genes, promoters ($\pm 3$ kb of TSS), introns, and distal intergenic regions.

#### Data Processing & Implementation
The peak calling pipeline is submitted to the HPC cluster using the **SLURM scheduler** (default: 4 GB memory, 1 CPU, 6-hour runtime). Following the generation of narrowPeak files, downstream Peak Quality Control (QC) and Genomic Annotation are performed within the R/Bioconductor environment (using packages such as ChIPseeker or ChIPpeakAnno).

#### Step 1: Peak Calling
MACS2 identifies regions of significant chromatin accessibility.

In [ ]:
# Run MACS2
conda activate macs2

In [ ]:
#!/bin/bash

# Create the output directory
mkdir -p macs2

#SBATCH --cpus-per-task=8
#SBATCH --mem=16G                
#SBATCH --output=logs/macs2_%A_%a.out
#SBATCH --error=logs/macs2_%A_%a.err

# Define groups: "path/to/file.bam:GroupName"
# You can add or remove groups by adding/deleting lines in this array
bam_files=(
    "./split_bam/Group1_shifted.bam:Group1"
    "./split_bam/Group2_shifted.bam:Group2"
    "./split_bam/Group3_shifted.bam:Group3"
    "./split_bam/Group4_shifted.bam:Group4"
    "./split_bam/Group4_shifted.bam:Group5"
)

# Iterate over the bam files and call peaks
# -g hs: Genome size for human (2.7e9)
# --keep-dup all: Retains all reads (standard for ATAC-seq if deduplicated previously)
# -f BAMPE: Uses paired-end insert sizes for better peak resolution
for entry in "${bam_files[@]}"; do
    IFS=":" read -r input_bam sample_name <<< "$entry"
    macs2 callpeak -t "$input_bam" -g hs --keep-dup all --cutoff-analysis -f BAMPE --outdir "./macs2/$sample_name" -n "$sample_name" 2> "./macs2/${sample_name}_macs2.log"
done

In [ ]:
# Run MultiQC
conda activate python3.7
multiqc ./macs2/. -o ./macs2/
conda deactivate 

| Sample Name | Fragment Length (bp) | Number of Peaks |
| :--- | :---: | :---: |
| **Group 1** | 333 | 85,891 |
| **Group 2** | 328 | 92,135 |
| **Group 3** | 325 | 90,956 |
| **Group 4** | 324 | 91,429 |
| **Group 5** | 332 | 89,321 |

* **High Consistency**: Fragment lengths (324–333 bp) and peak counts (~86k–92k) are remarkably uniform across all groups, indicating high technical reproducibility.
* **Robust Signal**: Achieving ~90,000 peaks per sample suggests a comprehensive capture of the open chromatin landscape with high sensitivity.
* **Pipeline Validation***: Stable metrics confirm that BAMPE mode and --binSize 10 optimizations preserved data integrity while ensuring computational efficiency.

#### Step 2: Peak QC (R Script)
Peak sets are imported into R for statistical filtering.

In [ ]:
# Load required libraries for peak QC, genomic manipulation, and visualization
library(ChIPQC)
library(rtracklayer)
library(DT)
library(tidyverse)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)

# Import ENCODE Blacklist regions (high-signal artifacts/repetitive regions)
# Reference: ENCSR636HFF - PMID 31249361
blkList <- import.bed("ENCFF356LFX.bed.gz")

# Locate all MACS2 .narrowPeak files generated in the previous step
openRegionPeaks <- list.files(path = "macs2", recursive = T, pattern = ".narrowPeak", full.names = T)

# Initialize list to store QC results for each sample
qcRes <- NULL

# Iterate through BAM files to calculate ChIP-seq/ATAC-seq specific QC metrics
for (i in 1:length(bamfile)) {
qcRes[[i]] <- ChIPQCsample(reads = bamfile[i],                   # Filtered BAM file
                           peaks = openRegionPeaks[i],           # Corresponding MACS2 peak file
                           annotation = "hg38",                  # Reference genome                                   
                           chromosomes = NULL,                   # Process all chromosomes
                           blacklist = blkList)                  # Regions to flag for noise calculation
  
}

# Extract 'Reads in Peaks' (RiP%) and 'Reads in Blacklist' (RiBL%)
# High RiP% (>20%) and low RiBL% (<1%) indicate high-quality libraries
lapply(qcRes, function(x) QCmetrics(x)[c("RiBL%", "RiP%")])

| Group   | RiBL (%) | RiP (%) |
|---------|----------|---------|
| Group 1 | 0.0449   | 65.40   |
| Group 2 | 0.0467   | 65.30   |
| Group 3 | 0.0458   | 66.70   |
| Group 4 | 0.0480   | 66.50   |
| Group 5 | 0.0433   | 69.10   |

These metrics indicate excellent data quality with a high signal-to-noise ratio across all groups.

* **Exceptional Signal (RiP%)**: Values between 65% and 69% are outstanding (typical high-quality samples are >20%), proving that the vast majority of the reads are mapping to functional open chromatin.
* **Minimal Noise (RiBL%)**: Percentages < 0.05% confirm negligible interference from problematic genomic regions (blacklist), ensuring the results are biologically genuine and not technical artifacts.

In [ ]:
## Calculate percentage of duplicated reads identified by ChIPQC
(unlist(lapply(qcRes, function(x) flagtagcounts(x)["DuplicateByChIPQC"]))/ 
    unlist(lapply(qcRes, function(x) flagtagcounts(x)["Mapped"])))*100

| Group   | DuplicateByChIPQC |
|---------|-------------------|
| Group 1 | 12.32             |
| Group 2 | 12.33             |
| Group 3 | 12.69             |
| Group 4 | 12.32             |
| Group 5 | 12.42             |

* **Consistency**: The near-identical values across all five groups demonstrate high technical reproducibility in the library preparation workflow.
* **Reliable Quantitation**: Low duplication ensures that the signal is derived from many unique transposition events rather than a few PCR-inflated reads

In [ ]:
# Convert QC results to GRanges objects for downstream manipulation
MacsCalls <- lapply(qcRes, granges)
unlist(lapply(MacsCalls, length))

# Remove peaks that overlap with ENCODE blacklist regions to improve signal-to-noise
MacsCalls <- lapply(MacsCalls, function(x) x[!x %over% blkList])

# Label the list for organized downstream analysis
names(MacsCalls) <- bamfile.label

# Verify final peak counts after filtering
cat("Peak counts after blacklist removal:\n")
unlist(lapply(MacsCalls, length))

| Group   | Number of Peaks |
|---------|-----------------|
| Group 1 | 92131           |
| Group 2 | 85887           |
| Group 3 | 89317           |
| Group 4 | 91425           |
| Group 5 | 90952           |

These peak counts (~86k–92k) represent a highly successful ATAC-seq experiment with robust genomic coverage.

* **High Sensitivity**: Identifying ~90k peaks per group is ideal for human samples, indicating the transposition was efficient enough to capture a vast landscape of both promoters and distal enhancers.
* **Technical Consistency**: The counts are remarkably stable across all five groups, which minimizes potential bias in the downstream "consensus peak set" and ensures fair comparisons during differential analysis.

#### Step 3: Genomic Annotations (R Script)
Validated peaks are mapped to the hg38 reference genome using ChIPseeker, categorizing the open chromatin landscape into functional features such as promoters, introns, and intergenic regions.

In [ ]:
# Load libraries for genomic feature annotation and reference genome data
library(ChIPseeker)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
library(BSgenome.Hsapiens.UCSC.hg38)

# Annotate peaks using the UCSC hg38 database
# tssRegion: Defines the promoter as ±3 kb around the Transcription Start Site
peakAnnoList <- lapply(MacsCalls, annotatePeak, TxDb = TxDb.Hsapiens.UCSC.hg38.knownGene,
                       tssRegion=c(-3000, 3000))

# Visualize the distribution of genomic features (Promoter, Intron, Distal Intergenic, etc.)
# This bar plot shows if your ATAC-seq signal is predominantly in promoters.
plotAnnoBar(peakAnnoList)

<img src="https://www.dropbox.com/scl/fi/hg288j5vnffj0zun58eej/Rplot-plot_annobar-ATAC.png?rlkey=ryt3x5qh8v4in9esf1wds4loj&st=29wcklmj&raw=1" width="600" alt="plot_annotation">

The Feature Distribution plot shows a highly consistent genomic landscape across all experimental conditions, with nearly identical proportions of regulatory elements.
* **Promoter Enrichment**: All groups show that approximately 30% of peaks are located within 3kb of a Transcription Start Site (TSS).
* **Distal Elements**: Roughly 25% of peaks fall into Distal Intergenic regions, and another 25% map to Introns, representing a rich set of potential enhancers that may be driving condition-specific responses.

#### Summary of Observations

* **Robust Peak Identification**: MACS2 identified between 85,891 and 92,135 peaks per sample. This high sensitivity indicates a comprehensive capture of the open chromatin landscape across all groups.
* **Exceptional Signal-to-Noise Ratio**: The Fraction of Reads in Peaks (FRiP) reached outstanding levels between 65% and 69%, significantly exceeding the standard 20% quality threshold. Minimal noise was confirmed by Reads in Blacklist (RiBL) values of <0.05%.
* **Technical Reproducibility**: Fragment lengths (324–333 bp) and duplication rates (~12.4%) were remarkably consistent across all five experimental groups, validating the library preparation and sequencing depth.
* **Genomic Distribution**: Approximately 30% of peaks are enriched at promoters ($\pm 3$ kb of TSS), while 50% are distributed across introns and distal intergenic regions, providing a rich set of potential enhancers for downstream analysis.